# 🚀 ZAIC 2026 DATAMATE — HUẤN LUYỆN LORA SFT TRÊN GPU A100

> **Mục tiêu:** Huấn luyện mô hình LoRA Adapter cho `Qwen/Qwen2.5-Coder-7B-Instruct` trên tập dữ liệu đã được kiểm chứng 100% ground-truth SQL để mô hình học chính xác cấu trúc JOIN và nghiệp vụ của các shop Việt Nam.

---

### 1. Dọn dẹp VRAM & Cài đặt Thư viện Huấn luyện

In [ ]:
# Dọn dẹp tiến trình vLLM cũ để giải phóng trọn vẹn 40GB VRAM
!pkill -f vllm || true
!pkill -f cloudflared || true

import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Cài đặt thư viện huấn luyện SFT tối ưu
!pip install -q trl peft transformers accelerate datasets

### 2. Nạp Tập Dữ Liệu Huấn luyện (train_sft.jsonl)
> Hãy tải file `train_sft.jsonl` từ máy tính lên mục Files (📁) của Colab trước khi chạy cell này.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/train_sft.jsonl", split="train")
print(f"📦 Đã nạp thành công {len(dataset)} mẫu huấn luyện SFT đã được kiểm chứng!")
print("Mẫu đầu tiên:", dataset[0]["id"], dataset[0]["business_id"])

### 3. Nạp Tokenizer & Mô hình Qwen2.5-Coder-7B (bfloat16) kèm Cấu hình LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
print(f"⏳ Đang nạp mô hình nền {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Cấu hình LoRA Adapter (Rank 16, Alpha 32)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 4. Tiến hành Huấn luyện LoRA SFT trên A100 (Khoảng 10–15 phút)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen2.5-coder-7b-lora-datamate",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Bắt đầu quá trình huấn luyện LoRA...")
trainer.train()

OUTPUT_DIR = "/content/qwen2.5-coder-7b-lora-datamate"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"🎉 Huấn luyện hoàn tất! Adapter đã được lưu tại: {OUTPUT_DIR}")

### 5. Khởi động lại vLLM Server phục vụ Mô hình LoRA & Mở Cloudflare Tunnel

In [ ]:
import subprocess, time, re

# 1. Khởi động vLLM kèm LoRA Module
print("🚀 Đang khởi động vLLM Server kèm LoRA Adapter 'datamate'...")
!vllm serve Qwen/Qwen2.5-Coder-7B-Instruct --enable-lora --lora-modules datamate=/content/qwen2.5-coder-7b-lora-datamate --port 8000 --max-model-len 8192 --gpu-memory-utilization 0.90 --dtype bfloat16 --trust-remote-code > /tmp/vllm.log 2>&1 &

# Đợi vLLM nạp model và adapter
import urllib.request
for i in range(24):
    time.sleep(5)
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                print(f"✅ vLLM LoRA Server đã sẵn sàng sau {(i+1)*5}s!")
                break
    except Exception:
        print(f"⏳ Đang nạp LoRA vào GPU (đã đợi {(i+1)*5}s)...")

# 2. Mở đường hầm Cloudflare Tunnel
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stderr.readline()
    m = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if m:
        tunnel_url = m.group(1)
        break
    time.sleep(0.1)

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 API ENDPOINT VLLM LORA SẴN SÀNG!")
    print(f"👉 Base URL: https://{tunnel_url}/v1")
    print("👉 Model Name: datamate")
    print("="*70 + "\n")